In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Klaim.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/sample_submission.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Polis.csv


# DATA FOUNDATION

IMPORT & LOAD DATA

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"

klaim = pd.read_csv(BASE_PATH + "Data_Klaim.csv")
polis = pd.read_csv(BASE_PATH + "Data_Polis.csv")

CLEAN COLUMN NAMES

In [3]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

klaim = clean_columns(klaim)
polis = clean_columns(polis)

klaim = klaim.drop_duplicates().reset_index(drop=True)
polis = polis.drop_duplicates().reset_index(drop=True)

DATE PARSING

In [4]:
for col in klaim.columns:
    if "tanggal" in col:
        klaim[col] = pd.to_datetime(klaim[col], errors="coerce")

for col in polis.columns:
    if "tanggal" in col:
        polis[col] = pd.to_datetime(polis[col], errors="coerce")

BASIC CLEANING

In [5]:
klaim = klaim.dropna(subset=["nomor_polis", "tanggal_pasien_masuk_rs"])
klaim["nominal_klaim_yang_disetujui"] = klaim["nominal_klaim_yang_disetujui"].fillna(0)

low_q = klaim["nominal_klaim_yang_disetujui"].quantile(0.005)
high_q = klaim["nominal_klaim_yang_disetujui"].quantile(0.995)

klaim["nominal_klaim_yang_disetujui"] = \
    klaim["nominal_klaim_yang_disetujui"].clip(low_q, high_q)

MERGE DATA

In [6]:
df = klaim.merge(polis, on="nomor_polis", how="left")

SERVICE MONTH

In [7]:
df["year_month"] = df["tanggal_pasien_masuk_rs"].dt.to_period("M")

TRUE EXPOSURE

In [8]:
exposure_monthly = (
    df.groupby("year_month")
      .agg(active_policies=("nomor_polis","nunique"))
      .reset_index()
      .sort_values("year_month")
)

df = df.merge(exposure_monthly, on="year_month", how="left")

BUILD MONTHLY CORE TABLE

In [9]:
monthly = (
    df.groupby("year_month")
      .agg(
          frequency=("nomor_polis","count"),
          total_claim=("nominal_klaim_yang_disetujui","sum"),
          exposure=("active_policies","mean")
      )
      .reset_index()
      .sort_values("year_month")
      .reset_index(drop=True)
)

DERIVED METRICS

In [10]:
monthly["severity"] = (
    monthly["total_claim"] /
    monthly["frequency"].replace(0,np.nan)
)

monthly["claim_rate"] = (
    monthly["frequency"] /
    monthly["exposure"].replace(0,np.nan)
)

LOG DOMAIN FEATURES

In [11]:
monthly["log_total"] = np.log1p(monthly["total_claim"])
monthly["log_freq"]  = np.log1p(monthly["frequency"])
monthly["log_sev"]   = np.log1p(monthly["severity"])
monthly["log_rate"]  = np.log1p(monthly["claim_rate"])

VOLATILITY FEATURES

In [12]:
monthly["roll6"] = monthly["total_claim"].rolling(6, min_periods=3).mean()
monthly["std6"]  = monthly["total_claim"].rolling(6, min_periods=3).std()

monthly["vol_ratio"] = monthly["std6"] / monthly["roll6"]

monthly["high_vol_regime"] = (
    monthly["vol_ratio"] > monthly["vol_ratio"].median()
).astype(int)

TIME FEATURES

In [13]:
monthly["month"] = monthly["year_month"].dt.month
monthly["month_sin"] = np.sin(2*np.pi*monthly["month"]/12)
monthly["month_cos"] = np.cos(2*np.pi*monthly["month"]/12)
monthly["month_index"] = np.arange(len(monthly))

SAFE LAGS

In [14]:
for col in ["log_total","log_freq","log_sev","log_rate"]:
    monthly[f"{col}_lag1"] = monthly[col].shift(1)
    monthly[f"{col}_lag2"] = monthly[col].shift(2)
    monthly[f"{col}_lag3"] = monthly[col].shift(3)

    monthly[f"{col}_roll3"] = monthly[col].shift(1).rolling(3).mean()

DROP EARLY MONTHS

In [15]:
monthly = monthly.dropna().reset_index(drop=True)

FINAL CHECK

In [16]:
print("Monthly shape:", monthly.shape)
print("Unique months:", monthly["year_month"].nunique())
print("Vol regime ratio:", round(monthly["high_vol_regime"].mean(),3))

Monthly shape: (16, 34)
Unique months: 16
Vol regime ratio: 0.438


# TIME-SERIES DATASET ENGINEERING

ENSURE REQUIRED SEGMENT COLUMNS

In [17]:
# Care Type
if "care_type" not in df.columns:
    if "inpatient_outpatient" in df.columns:
        df["care_type"] = (
            df["inpatient_outpatient"]
            .astype(str)
            .str.upper()
            .str.strip()
        )
    else:
        df["care_type"] = "UNKNOWN"

df["care_type"] = df["care_type"].fillna("UNKNOWN")


# Cashless
if "is_cashless" not in df.columns:
    if "reimburse_cashless" in df.columns:
        rc = df["reimburse_cashless"].astype(str).str.upper().str.strip()
        df["is_cashless"] = rc.eq("C").astype(int)
    else:
        df["is_cashless"] = 0


# RS Bucket
if "rs_bucket" not in df.columns:
    if "lokasi_rs" in df.columns:
        loc = df["lokasi_rs"].astype(str).str.upper().str.strip()
        df["rs_bucket"] = np.select(
            [
                loc.eq("INDONESIA"),
                loc.eq("SINGAPORE"),
                loc.eq("MALAYSIA")
            ],
            ["ID","SG","MY"],
            default="OTHER"
        )
    else:
        df["rs_bucket"] = "OTHER"

df["rs_bucket"] = df["rs_bucket"].fillna("OTHER")


# Plan Code
if "plan_code" not in df.columns:
    df["plan_code"] = "UNKNOWN"

df["plan_code"] = df["plan_code"].fillna("UNKNOWN")

DEFINE SEGMENT COLUMNS

In [18]:
seg_cols = ["plan_code","care_type","is_cashless","rs_bucket"]

BUILD SEGMENT MONTHLY TABLE

In [19]:
seg_monthly = (
    df.groupby(["year_month"] + seg_cols)
      .agg(
          frequency=("nomor_polis","count"),
          total_claim=("nominal_klaim_yang_disetujui","sum"),
          exposure=("nomor_polis","nunique")
      )
      .reset_index()
      .sort_values(seg_cols + ["year_month"])
      .reset_index(drop=True)
)

TARGET VARIABLES

In [20]:
seg_monthly["severity"] = (
    seg_monthly["total_claim"] /
    seg_monthly["frequency"].replace(0, np.nan)
)

seg_monthly["log_total"] = np.log1p(seg_monthly["total_claim"])
seg_monthly["log_freq"]  = np.log1p(seg_monthly["frequency"])
seg_monthly["log_sev"]   = np.log1p(seg_monthly["severity"])

CALENDAR FEATURES

In [21]:
seg_monthly["month"] = seg_monthly["year_month"].dt.month
seg_monthly["month_sin"] = np.sin(2*np.pi*seg_monthly["month"]/12)
seg_monthly["month_cos"] = np.cos(2*np.pi*seg_monthly["month"]/12)

LAG FEATURES

In [22]:
for col in ["log_total","log_freq","log_sev"]:
    
    seg_monthly[f"{col}_lag1"] = \
        seg_monthly.groupby(seg_cols)[col].shift(1)
    
    seg_monthly[f"{col}_lag2"] = \
        seg_monthly.groupby(seg_cols)[col].shift(2)
    
    seg_monthly[f"{col}_lag3"] = \
        seg_monthly.groupby(seg_cols)[col].shift(3)

    seg_monthly[f"{col}_roll3"] = \
        seg_monthly.groupby(seg_cols)[col] \
        .transform(lambda x: x.shift(1).rolling(3).mean())

MOMENTUM FEATURE

In [23]:
seg_monthly["momentum_total"] = (
    seg_monthly["log_total_lag1"] -
    seg_monthly["log_total_lag2"]
)

SEGMENT WEIGHT

In [24]:
seg_monthly["seg_weight"] = (
    seg_monthly["frequency"] /
    seg_monthly.groupby("year_month")["frequency"].transform("sum")
).fillna(0)

SAFE TRAIN WINDOW

In [25]:
seg_model = seg_monthly[
    seg_monthly["log_total_lag3"].notna()
].reset_index(drop=True)

seg_model = seg_model.fillna(0)

FINAL CHECK

In [26]:
print("COMPACT PANEL SHAPE:", seg_model.shape)
print("Unique segments:", seg_model[seg_cols].drop_duplicates().shape[0])
print("Columns:", len(seg_model.columns))

COMPACT PANEL SHAPE: (414, 29)
Unique segments: 41
Columns: 29


# MODEL DEVELOPMENT

IMPORT & SETUP

In [27]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

MAPE FUNCTION

In [28]:
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(
        np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
    ) * 100

BUILD MONTHLY TABLE

In [29]:
monthly = (
    df.groupby("year_month")
      .agg(
          frequency=("claim_id","count"),
          total_claim=("nominal_klaim_yang_disetujui","sum"),
          exposure=("active_policies","first")
      )
      .reset_index()
      .sort_values("year_month")
      .reset_index(drop=True)
)

monthly["severity"] = (
    monthly["total_claim"] /
    monthly["frequency"].replace(0,np.nan)
)

monthly["claim_rate"] = (
    monthly["frequency"] /
    monthly["exposure"].replace(0,np.nan)
)

TRAIN / VALID SPLIT

In [30]:
train = monthly.iloc[:-4].copy()
valid = monthly.iloc[-4:].copy()

sim_df = train.copy()

RECURSIVE FORECAST LOOP

In [31]:
pred_total = []
pred_freq  = []
pred_sev   = []

for i in range(4):

    train_sim = sim_df.copy()

    # DIRECT TOTAL MODEL (ETS)
    try:
        model_total = ExponentialSmoothing(
            np.log1p(train_sim["total_claim"]),
            trend="add",
            damped_trend=True,
            seasonal=None
        ).fit()

        total_pred = np.expm1(
            model_total.forecast(1).iloc[0]
        )
    except:
        total_pred = train_sim["total_claim"].iloc[-1]

    # SOFT SHRINK
    total_pred = (
        0.8 * total_pred +
        0.2 * train_sim["total_claim"].tail(3).mean()
    )

    # DERIVE FREQUENCY
    exposure_next = train_sim["exposure"].iloc[-1]
    rate_recent = train_sim["claim_rate"].tail(3).mean()

    freq_pred_i = max(
        rate_recent * exposure_next,
        1
    )

    sev_pred_i = total_pred / freq_pred_i

    # STORE PREDICTIONS
    pred_total.append(total_pred)
    pred_freq.append(freq_pred_i)
    pred_sev.append(sev_pred_i)

    # UPDATE RECURSIVE DATAFRAME
    new_row = {
        "frequency": freq_pred_i,
        "total_claim": total_pred,
        "exposure": exposure_next,
        "severity": sev_pred_i,
        "claim_rate": rate_recent
    }

    sim_df = pd.concat(
        [sim_df, pd.DataFrame([new_row])],
        ignore_index=True
    )

VALIDATION RESULTS

In [32]:
print("STRUCT MAPE Frequency :",
      round(mape(valid["frequency"], pred_freq),4))

print("STRUCT MAPE Total     :",
      round(mape(valid["total_claim"], pred_total),4))

print("STRUCT MAPE Severity  :",
      round(mape(valid["severity"], pred_sev),4))

print("Estimated Score           :",
      round(np.mean([
          mape(valid["frequency"], pred_freq),
          mape(valid["total_claim"], pred_total),
          mape(valid["severity"], pred_sev)
      ]),4))

STRUCT MAPE Frequency : 8.4632
STRUCT MAPE Total     : 5.0886
STRUCT MAPE Severity  : 6.7861
Estimated Score           : 6.7793


# TOTAL CLAIM OPTIMIZATION & VALIDATION, OPTUNA

INSTALL & IMPORT

In [33]:
!pip install -q optuna lightgbm

import optuna
import numpy as np
import pandas as pd
import lightgbm as lgb
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

STRICT MAPE FUNCTION

In [34]:
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(
        np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
    )

BUILD MONTHLY MODEL TABLE

In [35]:
monthly = (
    df.groupby("year_month")
      .agg(
          total_claim=("nominal_klaim_yang_disetujui","sum")
      )
      .reset_index()
      .sort_values("year_month")
      .reset_index(drop=True)
)

monthly["log_total"] = np.log1p(monthly["total_claim"])

monthly["month"] = monthly["year_month"].dt.month
monthly["month_sin"] = np.sin(2*np.pi*monthly["month"]/12)
monthly["month_cos"] = np.cos(2*np.pi*monthly["month"]/12)

for lag in [1,2,3]:
    monthly[f"log_lag{lag}"] = monthly["log_total"].shift(lag)

monthly["log_roll3"] = (
    monthly["log_total"]
    .shift(1)
    .rolling(3)
    .mean()
)

monthly = monthly.dropna().reset_index(drop=True)

DEFINE FEATURES

In [36]:
features = [
    "month_sin","month_cos",
    "log_lag1","log_lag2","log_lag3","log_roll3"
]

TRAIN / VALID SPLIT

In [37]:
train_full = monthly.iloc[:-4].copy()
valid_full = monthly.iloc[-4:].copy()

OBJECTIVE FUNCTION

In [38]:
def objective(trial):

    alpha  = trial.suggest_float("alpha", 0.4, 0.9)
    lr     = trial.suggest_float("lr", 0.005, 0.03)
    leaves = trial.suggest_int("leaves", 3, 10)
    shrink = trial.suggest_float("shrink", 0.7, 0.98)

    sim_df = train_full.copy()
    preds = []

    for step in range(4):

        sub_train = sim_df.copy()

        # ---------------- ETS ----------------
        try:
            ets = ExponentialSmoothing(
                sub_train["log_total"],
                trend="add",
                damped_trend=True,
                seasonal=None
            ).fit()

            pred_ets = np.expm1(
                ets.forecast(1).iloc[0]
            )
        except:
            pred_ets = sub_train["total_claim"].iloc[-1]

        # ---------------- LIGHTGBM ----------------
        model = lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=lr,
            num_leaves=leaves,
            min_data_in_leaf=4,
            verbosity=-1,
            random_state=42
        )

        model.fit(
            sub_train[features],
            sub_train["log_total"]
        )

        # ---------------- BUILD NEXT FEATURE ----------------
        next_row = sub_train.iloc[-1:].copy()
        next_month = next_row["month"].values[0] % 12 + 1

        new_feat = {
            "month_sin": np.sin(2*np.pi*next_month/12),
            "month_cos": np.cos(2*np.pi*next_month/12),
            "log_lag1": sub_train["log_total"].iloc[-1],
            "log_lag2": sub_train["log_total"].iloc[-2],
            "log_lag3": sub_train["log_total"].iloc[-3],
            "log_roll3": sub_train["log_total"].tail(3).mean()
        }

        X_new = pd.DataFrame([new_feat])
        pred_ml = np.expm1(model.predict(X_new)[0])

        # ---------------- HYBRID ----------------
        pred = alpha * pred_ets + (1-alpha) * pred_ml

        median_anchor = sub_train["total_claim"].tail(3).median()
        pred = shrink * pred + (1-shrink) * median_anchor

        preds.append(pred)

        # ---------------- UPDATE RECURSIVE ----------------
        new_row = {
            "year_month": None,
            "total_claim": pred,
            "log_total": np.log1p(pred),
            "month": next_month,
            "month_sin": new_feat["month_sin"],
            "month_cos": new_feat["month_cos"]
        }

        sim_df = pd.concat(
            [sim_df, pd.DataFrame([new_row])],
            ignore_index=True
        )

    return mape(valid_full["total_claim"], preds)

RUN OPTUNA

In [39]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=150)

print("\nBest Params:", study.best_params)
print("Best 4M MAPE:", round(study.best_value*100,4), "%")

[I 2026-02-25 22:16:14,775] A new study created in memory with name: no-name-4fe26866-cc7d-4144-a850-62cf2b93cfed
[I 2026-02-25 22:16:15,268] Trial 0 finished with value: 0.05309006368746969 and parameters: {'alpha': 0.6269929033372534, 'lr': 0.016319912433461918, 'leaves': 10, 'shrink': 0.7334554230320631}. Best is trial 0 with value: 0.05309006368746969.
[I 2026-02-25 22:16:15,661] Trial 1 finished with value: 0.05229008686131617 and parameters: {'alpha': 0.6643242631344071, 'lr': 0.022277613796878197, 'leaves': 3, 'shrink': 0.8020378857905605}. Best is trial 1 with value: 0.05229008686131617.
[I 2026-02-25 22:16:16,037] Trial 2 finished with value: 0.0501050825436952 and parameters: {'alpha': 0.464463641861829, 'lr': 0.015589036468348864, 'leaves': 10, 'shrink': 0.7450409257983946}. Best is trial 2 with value: 0.0501050825436952.
[I 2026-02-25 22:16:16,488] Trial 3 finished with value: 0.04900269674747302 and parameters: {'alpha': 0.539850672546291, 'lr': 0.0243089644594808, 'leaves


Best Params: {'alpha': 0.40031494427713477, 'lr': 0.027908008600093993, 'leaves': 3, 'shrink': 0.9739549827322527}
Best 4M MAPE: 2.3431 %


# TEST PREDICTION & KAGGLE SUBMISSION

IMPORT & LOAD SUBMISSION

In [40]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
sample_sub = pd.read_csv(BASE_PATH + "sample_submission.csv")

BUILD MONTHLY MODEL TABLE

In [41]:
monthly = (
    df.groupby("year_month")
      .agg(
          frequency=("claim_id","count"),
          total_claim=("nominal_klaim_yang_disetujui","sum"),
          exposure=("active_policies","first")
      )
      .reset_index()
      .sort_values("year_month")
      .reset_index(drop=True)
)

monthly["severity"] = (
    monthly["total_claim"] /
    monthly["frequency"].replace(0,np.nan)
)

monthly["claim_rate"] = (
    monthly["frequency"] /
    monthly["exposure"].replace(0,np.nan)
)

monthly["log_total"] = np.log1p(monthly["total_claim"])

FREEZE SAFE STATISTICS

In [42]:
EXPOSURE_SAFE = monthly["exposure"].tail(3).mean()
TOTAL_MEDIAN  = monthly["total_claim"].median()

PREPARE FUTURE PERIODS

In [43]:
sample_sub["year"]  = sample_sub["id"].str.split("_").str[0]
sample_sub["month"] = sample_sub["id"].str.split("_").str[1]
sample_sub["month_key"] = sample_sub["year"] + "-" + sample_sub["month"]

future_periods = (
    pd.PeriodIndex(sample_sub["month_key"], freq="M")
      .unique()
      .sort_values()
)

INITIALIZE RECURSIVE ENGINE

In [44]:
sim_df = monthly.copy()
predictions = {}

RECURSIVE FORECAST LOOP

In [45]:
for period in future_periods:

    train = sim_df.copy()

    # Conservative ETS total
    try:
        model_total = ExponentialSmoothing(
            np.log1p(train["total_claim"]),
            trend="add",
            damped_trend=True,
            seasonal=None
        ).fit()

        total_pred = np.expm1(
            model_total.forecast(1).iloc[0]
        )
    except:
        total_pred = train["total_claim"].iloc[-1]

    # Strong shrink to stability
    recent_mean = train["total_claim"].tail(3).mean()
    total_pred = (
        0.7 * total_pred +
        0.3 * recent_mean
    )

    # Conservative clamp
    lower = train["total_claim"].tail(6).min() * 0.9
    upper = train["total_claim"].tail(6).max() * 1.1

    total_pred = np.clip(total_pred, lower, upper)

    # Stable frequency
    rate_recent = train["claim_rate"].tail(3).mean()
    freq_pred = max(rate_recent * EXPOSURE_SAFE, 1)

    sev_pred = total_pred / freq_pred

    # UPDATE RECURSIVE DATAFRAME
    new_row = {
        "year_month": period,
        "frequency": freq_pred,
        "total_claim": total_pred,
        "exposure": EXPOSURE_SAFE,
        "severity": sev_pred,
        "claim_rate": rate_recent
    }

    sim_df = pd.concat(
        [sim_df, pd.DataFrame([new_row])],
        ignore_index=True
    )

    # STORE PREDICTIONS
    key = f"{period.year}_{str(period.month).zfill(2)}"

    predictions[f"{key}_Total_Claim"]      = total_pred
    predictions[f"{key}_Claim_Frequency"]  = freq_pred
    predictions[f"{key}_Claim_Severity"]   = sev_pred

BUILD SUBMISSION FILE

In [46]:
submission = sample_sub.copy()
submission["value"] = submission["id"].map(predictions)
submission = submission[["id","value"]]

submission.to_csv("submission.csv", index=False)

print("Submission created — Defensive Kaggle Safe")

Submission created — Defensive Kaggle Safe


In [47]:
print(submission.head(9))

                        id         value
0  2025_08_Claim_Frequency  2.462450e+02
1   2025_08_Claim_Severity  4.893961e+07
2      2025_08_Total_Claim  1.205113e+10
3  2025_09_Claim_Frequency  2.510911e+02
4   2025_09_Claim_Severity  4.794012e+07
5      2025_09_Total_Claim  1.203734e+10
6  2025_10_Claim_Frequency  2.481914e+02
7   2025_10_Claim_Severity  4.842043e+07
8      2025_10_Total_Claim  1.201753e+10
